In [1]:
import chess
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import random
from collections import deque

In [2]:
# Hyperparameters
STATE_SHAPE = (64,)
ACTION_SPACE = 4672
GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_MIN = 0.1
EPSILON_DECAY = 0.995
LEARNING_RATE = 0.001
BATCH_SIZE = 64
MEMORY_SIZE = 10000

In [3]:
# Create the DQN model
def create_dqn():
    model = models.Sequential([
        layers.Input(shape=STATE_SHAPE),
        layers.Dense(256, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(ACTION_SPACE, activation='linear')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='mse')
    return model

In [4]:
# Encode the chessboard into a flat array
def encode_board(board):
    board_matrix = np.zeros(64, dtype=int)
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        board_matrix[square] = piece.piece_type if piece.color else -piece.piece_type
    return board_matrix

# Select an action using epsilon-greedy policy
def select_action(model, state, legal_moves, epsilon):
    if np.random.rand() < epsilon:
        return random.choice(legal_moves)
    else:
        q_values = model.predict(state[np.newaxis], verbose=0)[0]
        legal_q_values = [q_values[move.to_square * 64 + move.from_square] for move in legal_moves]
        return legal_moves[np.argmax(legal_q_values)]

In [5]:
# Replay buffer for experience replay
class ReplayBuffer:
    def __init__(self, size):
        self.buffer = deque(maxlen=size)

    def add(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

In [6]:
# Train the model
def train_dqn(model, target_model, memory, batch_size):
    if len(memory.buffer) < batch_size:
        return

    batch = memory.sample(batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)

    states = np.array(states)
    next_states = np.array(next_states)
    targets = model.predict(states, verbose=0)

    # Update Q-values
    next_q_values = target_model.predict(next_states, verbose=0)
    for i in range(batch_size):
        action_index = actions[i].to_square * 64 + actions[i].from_square
        if dones[i]:
            targets[i, action_index] = rewards[i]
        else:
            targets[i, action_index] = rewards[i] + GAMMA * np.max(next_q_values[i])

    model.fit(states, targets, epochs=1, verbose=0, batch_size=batch_size)

In [7]:
# Reward system
def calculate_reward(board, previous_material, done):
    material_values = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 0,
    }

    def material_score(board):
        score = 0
        for piece_type, value in material_values.items():
            score += len(board.pieces(piece_type, chess.WHITE)) * value
            score -= len(board.pieces(piece_type, chess.BLACK)) * value
        return score

    if done:
        if board.is_checkmate():
            return 1.0 if board.turn == chess.BLACK else -1.0
        elif board.is_stalemate() or board.is_insufficient_material() or board.is_seventyfive_moves():
            return 0.5
        else:
            return 0

    current_material = material_score(board)
    material_reward = current_material - previous_material
    check_reward = 0.05 if board.is_check() else 0

    return material_reward * 0.1 + check_reward

In [ ]:
# Training loop
def train_chess_dqn_with_players():
    dqn = create_dqn()
    target_dqn = create_dqn()
    target_dqn.set_weights(dqn.get_weights())
    memory = ReplayBuffer(MEMORY_SIZE)

    epsilon = EPSILON_START

    for episode in range(1000):  # Train over 1000 games
        env = chess.Board()
        previous_material = 0
        total_reward = 0

        print(f"Starting Game {episode + 1}")

        while not env.is_game_over():
            print(env)
            print("Your Move (e.g., e2e4): ")
            human_move = input().strip()
            try:
                human_move = chess.Move.from_uci(human_move)
                if human_move not in env.legal_moves:
                    print("Illegal move. Try again.")
                    continue
            except ValueError:
                print("Invalid move format. Try again.")
                continue

            env.push(human_move)

            if env.is_game_over():
                break

            # AI's Turn
            state = encode_board(env)
            legal_moves = list(env.legal_moves)
            action = select_action(dqn, state, legal_moves, epsilon)
            env.push(action)

            reward = calculate_reward(env, previous_material, env.is_game_over())
            next_state = encode_board(env)
            done = env.is_game_over()

            memory.add((state, action, reward, next_state, done))
            train_dqn(dqn, target_dqn, memory, BATCH_SIZE)

            previous_material = material_score(env)
            total_reward += reward

        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

        if episode % 10 == 0:
            target_dqn.set_weights(dqn.get_weights())

        print(f"Game {episode + 1} ended with reward {total_reward}")

    return dqn

if __name__ == "__main__":
    trained_model = train_chess_dqn_with_players()

Starting Game 1
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
Your Move (e.g., e2e4): 


 'e2e4'


Invalid move format. Try again.
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
Your Move (e.g., e2e4): 
